In [23]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname
from utils import get_case_ids


pd.set_option("display.max_columns", None)
root_path = dirname(os.getcwd())
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/danbi/Projects/SANAGRAPH
/home/danbi/Projects/SANAGRAPH/data/datasets/original/
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/_processed/
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/


In [24]:
#ACT_TIME_ONLY = True 

In [25]:
ACT_TIME_ONLY = False

In [26]:
from pipeline import build_split_graphs, build_test_graphs_for_type, build_one_hot_encoders, group_by_case

In [27]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [28]:
list(datasets_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [29]:
dataset = "small_log_CZ"

In [30]:
dataset_graph_dir = data_dir_graphs + dataset + "/"
os.makedirs(dataset_graph_dir, exist_ok=True)

In [31]:
nan_methods = ["odd", "even", "random", "window","attr_level"]

masked_datasets = {key : pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_masked_{key}_all.csv") for key in nan_methods}


In [32]:
tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,time:timestamp
0,1,Activity A,0.0
1,1,Activity B,8.188966863648876
2,1,Activity C,8.881975184248867
3,1,Activity D,9.287394001418473
4,1,Activity E,9.575052927597383


In [33]:
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [34]:
if dataset == "BPI_Challenge_2012_W_Complete":
    tab_all["org:resource"] = tab_all["org:resource"].astype(np.str_)
    tab_train["org:resource"] = tab_train["org:resource"].astype(np.str_)
    tab_valid["org:resource"] = tab_valid["org:resource"].astype(np.str_)
    tab_test["org:resource"] = tab_test["org:resource"].astype(np.str_)

In [35]:
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [36]:
dataset_info

{'categorical': ['Activity'], 'numerical': ['time:timestamp']}

In [37]:
if ACT_TIME_ONLY:
    categorical_columns = ["Activity"]
    real_value_columns = ["time:timestamp"]
    dataset = f"{dataset}_AT_only"
else:
    categorical_columns = dataset_info["categorical"]
    real_value_columns = dataset_info["numerical"]

In [38]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    
    for k_m in masked_datasets:
        masked_datasets[k_m][k] = masked_datasets[k_m][k].astype("object")

In [39]:
from numpy import NaN
if dataset == "sp2020_CZ":
    tab_all["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_all["REPAIR_IN_TIME_5D"].values]
    tab_train["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_train["REPAIR_IN_TIME_5D"].values]
    tab_valid["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_valid["REPAIR_IN_TIME_5D"].values]
    tab_test["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in tab_test["REPAIR_IN_TIME_5D"].values]
    for k_m in masked_datasets:
        masked_datasets[k_m]["REPAIR_IN_TIME_5D"] = [float(x) if x is not NaN else x for x in masked_datasets[k_m]["REPAIR_IN_TIME_5D"].values]

In [40]:
for k in categorical_columns:
    print(f"{k} {tab_test[k].values.dtype}")

Activity object


In [41]:
dataset

'small_log_CZ'

In [42]:
from numpy import NaN
from math import log
if dataset == "BPI20_RequestForPayment_CZ":
    tab_all["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_all["case:RequestedAmount"].values]
    tab_train["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_train["case:RequestedAmount"].values]
    tab_valid["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_valid["case:RequestedAmount"].values]
    tab_test["case:RequestedAmount"] = [log(x) if x > 0 else 0. for x in tab_test["case:RequestedAmount"].values]
    for k_m in masked_datasets:
        masked_datasets[k_m]["case:RequestedAmount"] = [log(x) if x > 0 else 0. if x == 0 else x for x in masked_datasets[k_m]["case:RequestedAmount"].values]

### Prepare the graphs

In [43]:
from torch import tensor,int64, float32
from torch_geometric.data import HeteroData


In [44]:
from pipeline import MISSING_VALUE

In [45]:
import sklearn.preprocessing
from pipeline import get_one_hot_encoder

In [46]:
from pipeline import add_new_timestamp

In [47]:
from pipeline import get_node_features

In [48]:
from pipeline import compute_edges_indexs

In [49]:
dataset

'small_log_CZ'

In [50]:
def get_masked_trace(dataset_traces, cat_features, real_features, caseid):
    trace = (
        dataset_traces.query(f"CaseID == '{caseid}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
    
    
    
    if dataset != "bpi_2012_CZ" and dataset != 'bpi_2012_CZ_AT_only':
        mask = trace[trace.columns].isnull().apply(lambda x: all(x), axis=1)
    else:
        mask = trace[trace.columns].isnull().any(axis=1).values
        
    mask_index = [i for i in range(len(mask)) if mask[i]]
    
    for k in cat_features:
        for i in mask_index:
            trace.loc[i, k] = MISSING_VALUE
    
    for k in real_features:
        for i in mask_index:
            trace.loc[i, k] = -1
    
    return trace, mask
    
    

In [51]:
import torch 
from pipeline import get_masked_trace_dict_mask    

In [52]:
nan_methods[0]

'odd'

In [53]:
masked_datasets[nan_methods[0]]

,CaseID,Activity,time:timestamp
0,1,Activity A,0.000000
1,1,NaN,NaN
2,1,Activity C,8.881975
3,1,NaN,NaN
4,1,Activity E,9.575053
...,...,...,...
27995,2000,Activity I,9.980495
27996,2000,Activity N,9.980495
27997,2000,Activity L,9.980495
27998,2000,Activity H,10.134639


In [54]:

from copy import copy
import torch
from pipeline import build_prefixes_graph_from_trace

## Create the datasets

In [55]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [56]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

1200
400
400


In [57]:
#trace2 = (
#        masked_datasets["odd"].query(f"CaseID == '{case_train_ids[0]}'")
#        .reset_index()
#        .drop(columns="index")
#        .drop(columns="CaseID")
#    )
#trace2

In [58]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [59]:
for k in masked_datasets:
    masked_datasets[k]["CaseID"] = masked_datasets[k]["CaseID"].astype(np.str_)

In [60]:
encoders = build_one_hot_encoders(tab_all, categorical_columns)
grouped_train = group_by_case(tab_train)
grouped_valid = group_by_case(tab_valid)
grouped_test = group_by_case(tab_test)
grouped_masked = {method: group_by_case(df) for method, df in masked_datasets.items()}

In [61]:
#tab_all["REPAIR_IN_TIME_5D"]

In [62]:
#masked_datasets["even"]["REPAIR_IN_TIME_5D"]

In [63]:
#trace = (
#        tab_train.query(f"CaseID == '{case_train_ids[1]}'")
#        .reset_index()
#        .drop(columns="index")
#        .drop(columns="CaseID")
#    )
#trace 

In [64]:
#trace3 = add_new_timestamp(trace)
#trace3

In [65]:
#m_trace, mask = get_masked_trace_dict_mask(masked_datasets["attr_level"], categorical_columns, real_value_columns, case_train_ids[1], trace3["time:timestamp"].values)

In [66]:
#m_trace

In [67]:
#mask

In [68]:
#graphs = build_prefixes_graph_from_trace(tab_all, trace, categorical_columns, real_value_columns, case_train_ids[1])

In [69]:
#graphs[0].x_dict["time:timestamp"]

In [70]:
#graphs[0].y

In [71]:
#graphs[0].masks

In [72]:
#graphs[0].y["Activity"][graphs[0].masks["Activity"]]

In [73]:
from tqdm.notebook import tqdm

In [74]:
from pipeline import build_split_graphs
import pickle

print("Preparing training dataset...")

X_train = build_split_graphs(
    tab_all, tab_train, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_train_ids
)
with open(dataset_graph_dir + dataset + "_TRAIN_V2_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)
del X_train
print("Done!\n\n")


Preparing training dataset...


  0%|          | 0/1200 [00:00<?, ?it/s]

Done!




In [75]:
print("Preparing validation dataset...")

X_valid = build_split_graphs(
    tab_all, tab_valid, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_valid_ids
)
with open(dataset_graph_dir + dataset + "_VALID_V2_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)
del X_valid
print("Done!\n\n")

Preparing validation dataset...


  0%|          | 0/400 [00:00<?, ?it/s]

Done!




In [76]:
print("Preparing test dataset...")

X_test = build_split_graphs(
    tab_all, tab_test, categorical_columns, real_value_columns,
    masked_datasets, nan_methods, case_ids=case_test_ids
)
with open(dataset_graph_dir + dataset + "_TEST_V2_repair.pkl", "wb") as f:
    pickle.dump(X_test, f)
del X_test
print("Done!\n\n")

Preparing test dataset...


  0%|          | 0/400 [00:00<?, ?it/s]

Done!




In [77]:
from pipeline import build_test_graphs_for_type

In [78]:
def create_and_save_test(case: str):
    X_test = build_test_graphs_for_type(
        tab_all, tab_test, categorical_columns, real_value_columns,
        masked_datasets, nan_methods, mask_type=case, case_ids=case_test_ids
    )
    print(dataset_graph_dir + dataset + f"_TEST_V2_repair_{case}.pkl")
    with open(dataset_graph_dir + dataset + f"_TEST_V2_repair_{case}.pkl", "wb") as f:
        pickle.dump(X_test, f)

In [79]:
data_dir_graphs

'/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/'

In [80]:
create_and_save_test("even")

  0%|          | 0/400 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/small_log_CZ/small_log_CZ_TEST_V2_repair_even.pkl


In [81]:
create_and_save_test("odd")

  0%|          | 0/400 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/small_log_CZ/small_log_CZ_TEST_V2_repair_odd.pkl


In [82]:
create_and_save_test("random")

  0%|          | 0/400 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/small_log_CZ/small_log_CZ_TEST_V2_repair_random.pkl


In [83]:
create_and_save_test("window")

  0%|          | 0/400 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/small_log_CZ/small_log_CZ_TEST_V2_repair_window.pkl


In [84]:
create_and_save_test("attr_level")

  0%|          | 0/400 [00:00<?, ?it/s]

/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/small_log_CZ/small_log_CZ_TEST_V2_repair_attr_level.pkl
